In [1]:
%load_ext autoreload
%autoreload 2

import os
import warnings
import pandas as pd
import numpy as np
from scipy import signal
import math

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from matplotlib.ticker import MaxNLocator
import seaborn as sns
%matplotlib qt

import tools.particle_swarm_optimization as pso
import tools.particle_swarm_optimization_plot as pso_plt
import tools.peristimulus_time_histogram as psth

warnings.filterwarnings("ignore")


              -- N E S T --
  Copyright (C) 2004 The NEST Initiative

 Version: 3.7.0
 Built: Apr 15 2024 07:13:33

 This program is provided AS IS and comes with
 NO WARRANTY. See the file LICENSE for details.

 Problems or suggestions?
   Visit https://www.nest-simulator.org

 Type 'nest.help()' to find out more about NEST.



In [43]:
# read and proccees the lfp
ruta_principal = os.path.join(os.getcwd(), 'results/potjans_diesmann/')
carpetas = [nombre for nombre in os.listdir(ruta_principal)
            if os.path.isdir(os.path.join(ruta_principal, nombre))]

plt.close('all')

plot1, plot2, plot3 = True, True, True

aux=0
for carpeta in carpetas:
    folder = os.path.join(ruta_principal, carpeta)

    if aux == 1:
        continue
    aux=1    

    # read CSV into algo
    pdlfp = pd.read_csv(os.path.join(folder, 'lfp_layer_1.csv'), sep=' ', header=None, names=['lfp'])
    pdlfp2 = pd.read_csv(os.path.join(folder, 'lfp_layer_sph_1.csv'), sep=' ', header=None, names=['lfp'])
    
    lfp_signal = pdlfp['lfp'].to_numpy()
    fs = int(len(lfp_signal)/3)

    lfp_signal22 = pdlfp2['lfp'].to_numpy()
    fs2 = int(len(lfp_signal22)/3)

    # --- APLICA UN FILTRO PASO BAJO ---
    # Frecuencia de corte (e.g., 200 Hz)
    cutoff_freq = 200

    # Diseñar el filtro (Butterworth es una opción estándar)
    b, a = signal.butter(4, cutoff_freq, btype='low', fs=fs)

    # Aplicar el filtro a tu señal
    lfp_signal = signal.filtfilt(b, a, lfp_signal)

    cutoff_freq = 50
    # Diseñar el filtro (Butterworth es una opción estándar)
    b, a = signal.butter(4, cutoff_freq, btype='low', fs=fs)
    b2, a2 = signal.butter(4, cutoff_freq, btype='low', fs=fs2)

    # Aplicar el filtro a tu señal
    lfp_signal2 = signal.filtfilt(b, a, lfp_signal)
    lfp_signal22 = signal.filtfilt(b2, a2, lfp_signal22)

    time = np.arange(0, 3, 1/fs)
    time2 = np.arange(0, 3, 1/fs2)

    if plot1:
        fig, ax = plt.subplots(1, 1, layout='constrained', figsize=(10, 6), sharey=True)
        ax.plot(time, lfp_signal2)
        ax.plot(time2, lfp_signal22)
        ax.set_ylabel('LFP volt')
        ax.set_xlabel('Time (s)')
        ax.set_xlim(0, 3)
        y1, y2 = ax.get_ylim()

        ax.plot([1,1], [y1,y2],'--k')
        ax.plot([2,2], [y1,y2],'--k')
        ax.set_ylim(y1, y2)

        fig.suptitle('Optimización de Parámetros con PSO\nConectividad Lateral L2/3')
        plt.show()

    # epsctrum
    

    if plot2:
        freqs, psd = signal.welch(lfp_signal, fs, nperseg=fs)

        # Evitar log(0) añadiendo un valor muy pequeño si hay ceros en psd
        psd[psd == 0] = np.finfo(float).eps

        # --- CAMBIO AQUÍ ---
        # 1. Convertir la potencia a decibeles
        psd_db = 10 * np.log10(psd)
        #psd_db = psd
        
        plt.figure(figsize=(10, 5))
        
        # 2. Usar plt.plot() normal, ya que los datos ya están en escala logarítmica
        plt.plot(freqs, psd_db)
        
        plt.title('Espectro de Potencia (LFP en dB)')
        plt.xlabel('Frecuencia (Hz)')
        
        # 3. Actualizar la etiqueta del eje Y
        plt.ylabel('Densidad Espectral de Potencia (dB/Hz)')
        
        #plt.grid(linestyle='--', alpha=0.7)
        plt.xlim(0, 200)
        plt.ylim(-50, 40)
        plt.show()


    # power spectrum
    f, t, Sxx = signal.spectrogram(lfp_signal, fs, nperseg=10000, nfft=20000)
    Sxx_db = 10 * np.log10(Sxx + np.finfo(float).eps)

    if plot3:

        plt.figure(figsize=(12, 6))
        plt.pcolormesh(t, f, Sxx_db, shading='gouraud', vmin=-25)#, vmax=35)
        plt.title('Espectrograma (LFP)')
        plt.xlabel('Tiempo (s)')
        plt.ylabel('Frecuencia (Hz)')
        plt.colorbar(label='Potencia (dB)')
        
        # --- CAMBIO AQUÍ ---
        # Ajustamos el límite del eje Y para enfocarnos en el rango de LFP
        plt.ylim(0, 100)
        plt.plot([1, 1], [0, 100], '--k')
        plt.plot([2, 2], [0, 100], '--k')
        plt.show()



In [52]:
# read and proccees the lfp
ruta_principal = os.path.join(os.getcwd(), 'results/potjans_diesmann/')
carpetas = [nombre for nombre in os.listdir(ruta_principal)
            if os.path.isdir(os.path.join(ruta_principal, nombre))]

plt.close('all')

for carpeta in carpetas:
    folder = os.path.join(ruta_principal, carpeta) 

    for layer in range(4):
        # read CSV into algo
        pdlfp = pd.read_csv(os.path.join(folder, ('lfp_layer_'+str(layer+1)+'.csv')), sep=' ', header=None, names=['lfp'])

        pdlfp_sph = pd.read_csv(os.path.join(folder, ('lfp_layer_sph_'+str(layer+1)+'.csv')), sep=' ', header=None, names=['lfp'])
    
        lfp_signal = pdlfp['lfp'].to_numpy()
        lfp_signal_sph = pdlfp_sph['lfp'].to_numpy()

        fs = int(len(lfp_signal)/3)

        # Frecuencia de corte (e.g., 200 Hz)
        cutoff_freq = 200
        b, a = signal.butter(4, cutoff_freq, btype='low', fs=fs)

        # Aplicar el filtro a tu señal
        lfp_signal = signal.filtfilt(b, a, lfp_signal)
        lfp_signal_sph = signal.filtfilt(b, a, lfp_signal_sph)

        time = np.arange(0, 3, 1/fs)

        len_win = int(len(lfp_signal)/3)
        
        for etapa in range (3):

            freqs, psd = signal.welch(lfp_signal[etapa*len_win:(etapa+1)*len_win], fs, nperseg=fs)
            freqs, psd_sph = signal.welch(lfp_signal_sph[etapa*len_win:(etapa+1)*len_win], fs, nperseg=fs)

            # Evitar log(0) añadiendo un valor muy pequeño si hay ceros en psd
            psd[psd == 0] = np.finfo(float).eps
            psd_sph[psd_sph == 0] = np.finfo(float).eps
            
            psd = psd[freqs <= 200]
            psd_sph = psd_sph[freqs <= 200]

            # --- CAMBIO AQUÍ ---
            # # 1. Convertir la potencia a decibeles
            # psd_db = 10 * np.log10(psd)
            # psd_db_sph = 10 * np.log10(psd_sph)
            #psd_db = psd
            
            lfp_csv = [layer+1, 0, etapa] + psd.tolist()
            df = pd.DataFrame([lfp_csv])
            df.to_csv(os.path.join(ruta_principal, 'lfp.csv'), mode='a', index=False, header=False)

            lfp_csv = [layer+1, 1, etapa] + psd_sph.tolist()
            df = pd.DataFrame([lfp_csv])
            df.to_csv(os.path.join(ruta_principal, 'lfp.csv'), mode='a', index=False, header=False)
